# 04 — LightGBM and heroes

**Goal:** test two ideas the baseline (notebook 03, validation log loss **0.5489**) leaves out, one at a time, then score the final model on the **test split, once**.

1. **Heroes.** Which 6 heroes each team picked is known from minute 0, when the baseline is close to a coin flip.
2. **Interactions.** A linear model adds up independent effects. It can't learn "a soul lead matters more when you also hold the Titan". Tree models learn such interactions on their own.

**One-hot encoding.** A hero id is a label, not a number (hero 52 isn't "more" than hero 25), so each hero gets its own column: **+1** if Team0 picked it, **−1** if Team1 did, **0** if neither. It's a diff like the others: swapping teams negates it.

**Decision trees and gradient boosting.** A decision tree splits the data with yes/no questions ("net_worth_diff > 3,000?", "t > 1,500?") and predicts a value in each leaf. One small tree is crude. **Gradient boosting** builds hundreds of them in sequence, each one fitted to the errors the previous trees still make. LightGBM is a fast implementation. Two settings control complexity: `num_leaves` (how detailed one tree can be) and `min_child_samples` (how many rows a leaf needs, a guard against fitting noise).

**Early stopping.** Every extra tree fits train a bit better, but past some point it starts fitting noise. We stop adding trees when validation log loss hasn't improved for 100 rounds, which chooses the tree count on validation.

**Symmetry for trees.** The baseline is symmetric by construction (odd logit, no intercept). Trees aren't: a split "net_worth_diff > 2,000" has no mirror image built in. So we train on every snapshot **and** its team-swapped copy, and predict with logit(P) = ½·(g(x) − g(mirror x)), where g is the raw tree score. Swapping teams negates that exactly. Unlike in notebook 03, mirroring really changes the model here.

**The 2×2 comparison:** logistic regression or LightGBM, each with and without heroes, all on validation. The final pick follows the rule from notebook 03: the simplest model within 0.0005 log loss of the best.

**Before running:** `python -m deadlock_predictor.data.pipeline --skip-extract` (hero table), then `python -m deadlock_predictor.models.train_baseline` and `python -m deadlock_predictor.models.train_gbm` (about 40 minutes in total).

In [ ]:
import json
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from deadlock_predictor.data.assets import hero_name
from deadlock_predictor.features.snapshots import LABEL
from deadlock_predictor.models.baseline import weight_curves
from deadlock_predictor.models.data import (
    HEROES_PATH,
    add_heroes,
    load_split,
    predict_saved,
    snapshot_columns,
)
from deadlock_predictor.models.evaluate import reliability, scores, scores_by_time
from deadlock_predictor.models.gbm import mirror_features

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MODELS = ROOT / "models"
assert (MODELS / "final.json").exists(), "Run train_baseline and train_gbm first (see above)"
CELLS = ["baseline", "baseline_heroes", "gbm", "gbm_heroes"]
DESCRIBE = {
    "baseline": "LR, no heroes (rung 3)",
    "baseline_heroes": "LR + heroes",
    "gbm": "LightGBM, no heroes",
    "gbm_heroes": "LightGBM + heroes",
}
final = json.loads((MODELS / "final.json").read_text())
metrics = {c: json.loads((MODELS / f"{c}_metrics.json").read_text()) for c in CELLS}
saved = {c: joblib.load(MODELS / f"{c}.joblib") for c in CELLS}
columns = sorted({col for s in saved.values() for col in snapshot_columns(s["columns"])})
val = load_split("validation", columns + ["start_time"])
y_val = val[LABEL].to_numpy()
plt.rcParams["figure.figsize"] = (9, 4)
warnings.filterwarnings("ignore", message="The inplace keyword")
print(f"validation {len(val):,} rows; final choice: {final['choice']}")

## 1. Heroes

First, the raw signal: how often the team with each hero won, over train matches. (Names come from the deadlock-api assets endpoint when it's reachable; otherwise ids.)

In [ ]:
# Win rate of the team that picked each hero, in train matches.
train_matches = load_split("train", []).drop_duplicates("match_id")[["match_id", LABEL]]
picks = pd.read_parquet(HEROES_PATH).merge(train_matches, on="match_id")
picks["won"] = np.where(picks.sign == 1, picks[LABEL], ~picks[LABEL])
rates = picks.groupby("hero_id").won.agg(win_rate="mean", matches="size")
rates["se"] = np.sqrt(rates.win_rate * (1 - rates.win_rate) / rates.matches)
rates = rates.sort_values("win_rate")
fig, ax = plt.subplots(figsize=(9, 9))
ax.barh([hero_name(h) for h in rates.index], rates.win_rate - 0.5, xerr=2 * rates.se, left=0.5)
ax.axvline(0.5, color="grey", lw=0.8)
ax.set(title="Win rate of the team with each hero (train, ±2 standard errors)", xlabel="win rate")
ax.tick_params(axis="y", labelsize=7)
print(f"{len(rates)} heroes; win rates {rates.win_rate.min():.3f} to {rates.win_rate.max():.3f}")

Hero win rates span roughly 38% to 57%, and with 25k–250k matches per hero the error bars are only ±0.2–0.6 points: these are real differences, not noise. In the LR model, the strongest heroes are worth about 0.4 logit, as much as a 2,000-soul lead at 5 minutes, and that's known before the game starts.

Win rates alone don't separate a hero's strength from who plays it (a hero popular in high ranks, or with players who are better on average). The model doesn't need to know which: it only needs the pick to predict the outcome.

**A complication: new heroes.** The data covers one patch, but heroes can still be released during it:

In [ ]:
known = set(saved["gbm_heroes"]["hero_ids"])
val_matches = val.drop_duplicates("match_id")[["match_id", "start_time"]]
hero_table = pd.read_parquet(HEROES_PATH)
val_picks = hero_table[hero_table.match_id.isin(val_matches.match_id)]
new = sorted(set(val_picks.hero_id) - known)
with_new = set(val_picks.match_id[val_picks.hero_id.isin(new)])
day = val_matches.start_time.dt.tz_convert("UTC").dt.date
share = val_matches.match_id.isin(with_new).groupby(day).mean()
print(f"heroes in validation but never in train: {[hero_name(h) for h in new]}")
print(f"validation matches with one of them: {len(with_new) / len(val_matches):.1%}")
share.rename("share of matches with a new hero").to_frame().round(3)

Hero 86 was released on Oct 16, a few hours after validation starts, and is in **~84%** of validation matches from then on (everyone tries the new hero). It is never in train, so the train-only models give it no column: for them, a team with hero 86 has 5 known heroes. Checking validation separately, matches with hero 86 score about the same log loss as matches without it, so this costs less than expected. It matters more for calibration (section 4).

## 2. The 2×2 comparison (validation)

In [ ]:
table = pd.DataFrame(
    {c: {"model": DESCRIBE[c], **metrics[c]["best"]["validation"]} for c in CELLS}
).T
table["vs baseline"] = table.log_loss - table.loc["baseline", "log_loss"]
table

In [ ]:
grid = pd.DataFrame(
    [
        {"cell": c, **r["params"], "trees": r["trees"], "log_loss": r["validation"]["log_loss"]}
        for c in ("gbm", "gbm_heroes")
        for r in metrics[c]["grid"]
    ]
)
grid.round(5)

- **Heroes are the big gain:** −0.030 log loss (0.5489 → 0.5192 for logistic regression), more than everything in notebook 03's ladder after net worth. Accuracy rises from 70.2% to 72.8%.
- **Trees add almost nothing:** −0.0002 without heroes, −0.0009 with heroes. The baseline's time-varying weights already capture most of the structure, so there are few interactions left for trees to find.
- **The final pick is LightGBM + heroes.** It beats LR + heroes by 0.00085, just above our 0.0005 tolerance. That margin is small, and a little optimistic: LightGBM also used validation for early stopping, a mild form of fitting to it. The test score will tell whether the edge is real.
- **Tree settings barely matter** (spread 0.0004 within each cell). Small trees (31 leaves) with large leaves (2,000 rows) do best: a smooth, low-noise problem, where many simple trees beat fewer detailed ones.
- **For LR + heroes, rung 6 (hero weights vary with time) won**, 0.5192 vs 0.5201 for constant hero weights. Some heroes are early-game or late-game heroes (section 6).

## 3. Where each idea helps

In [ ]:
preds = {c: predict_saved(saved[c], val) for c in CELLS}
by_time = {c: scores_by_time(val.t.to_numpy(), y_val, p) for c, p in preds.items()}
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
for c, st in by_time.items():
    ax1.plot(st.index / 60, st.log_loss, marker=".", label=DESCRIBE[c])
    gain = by_time["baseline"].log_loss - st.log_loss
    ax2.plot(st.index / 60, gain, marker=".", label=DESCRIBE[c])
ax1.axhline(np.log(2), color="grey", ls="--", label="always 50%")
ax1.set(title="Validation log loss by game time", xlabel="minute", ylabel="log loss")
ax2.axhline(0, color="grey", lw=0.8)
ax2.set(title="Improvement over the baseline", xlabel="minute", ylabel="log loss saved")
ax1.legend()
at = [180, 540, 900, 1200, 1800, 2400, 3000]
pd.DataFrame({DESCRIBE[c]: st.log_loss.reindex(at) for c, st in by_time.items()}).rename(
    index=lambda t: f"{t // 60} min"
).round(4)

- **Heroes help at every minute**, by about 0.025–0.034. The biggest relative gain is early (3 min: 0.671 → 0.646), when there's little else to go on. Even at 50 minutes, team composition still says something the stats don't.
- **Trees help early and hurt late.** LightGBM + heroes beats LR + heroes up to ~15 minutes (0.593 vs 0.596 at 9 min), probably because hero combinations matter there (a tree can learn "hero A *with* hero B"), and it is worse after 35 minutes (0.405 vs 0.401 at 40 min). Long games are rare, and trees predict in steps from the few rows they saw, while the spline weights extend smoothly.
- That suggests a hybrid (LR late, trees early), but the gains are tiny and it would be one more choice fitted to validation.

## 4. Calibration

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0, 1], [0, 1], color="grey", ls="--")
rel = {}
for c in dict.fromkeys(["baseline", final["choice"]]):
    rel[c] = reliability(y_val, preds[c])
    ax.plot(rel[c].predicted, rel[c].observed, marker="o", label=DESCRIBE[c])
ax.set(title="Reliability (validation)", xlabel="predicted P(Team0 wins)", ylabel="observed")
ax.legend()
pd.concat(rel, axis=1).round(3)

**The hero models are less well calibrated:** ECE rose from 0.004 to 0.014. The pattern is overconfidence: when LightGBM + heroes says 25%, that team wins 28%; when it says 75%, they win 73%.

The cause is **drift in hero strength**. Comparing per-hero win rates, a hero's edge over 50% in validation is only about **0.84×** its edge in train (correlation 0.92: the same heroes are strong, just less so). Players adapt to strong heroes, the new hero changes matchups, and small balance updates land between major patches. A model fitted to train's hero edges overstates them later. The best single correction on validation would scale the logit by 0.95.

Hero 86 adds to this a little: calibration is worse in matches with it (ECE 0.015) than without (0.010).

**What we did about it:** before the test, the baseline and the final model are refit on train **+ validation** with the same settings (same C, same tree count). That's the usual last step once settings are fixed: validation's matches are the newest data before the test period, and hero 86 gets a weight. We did **not** fit a calibration correction on validation, since we couldn't then judge it on validation.

## 5. Is the tree model really symmetric?

On real data: predict 100k validation snapshots and their mirrors.

In [ ]:
sample = val.sample(100_000, random_state=0)
g = saved["gbm_heroes"]
X = add_heroes(sample, g["hero_ids"])[g["columns"]]
p = g["model"].predict_proba(X)[:, 1]
p_mirror = g["model"].predict_proba(mirror_features(X))[:, 1]
print(f"largest |P(x) + P(mirror x) − 1| on 100k rows: {np.abs(p + p_mirror - 1).max():.1e}")

The sum is 1 to machine precision (~1e-16), as the construction promises.

## 6. What drives the predictions

**SHAP values** split one prediction's logit into a contribution per feature that adds up exactly to the prediction; LightGBM computes them (`pred_contrib=True`), and we symmetrize them the same way as the logit. Averaging their size over many snapshots shows how much each feature moves predictions, at each stage of the game. All hero columns are summed into one "heroes" contribution.

In [ ]:
contrib = g["model"].contributions(X)
hero_cols = [c for c in contrib.columns if c.startswith("hero_")]
contrib["heroes (all 6 v 6)"] = contrib[hero_cols].sum(axis=1)
contrib = contrib.drop(columns=hero_cols)
minute = pd.cut(
    sample.t.astype(float) / 60,
    [0, 8, 15, 25, 35, 45, 200],
    labels=["3-8", "9-15", "16-25", "26-35", "36-45", "46+"],
)
importance = contrib.abs().groupby(minute, observed=True).mean().T
importance = importance.loc[importance.mean(axis=1).sort_values(ascending=False).index]
importance.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.set_prop_cycle(color=plt.cm.tab20.colors)
for name, row in importance.head(10).iterrows():
    ax.plot(row.index.astype(str), row.to_numpy(), marker="o", label=name.removesuffix("_diff"))
ax.set(
    title="Mean |contribution to the logit| by game time (LightGBM + heroes)",
    xlabel="minute",
    ylabel="mean |Δ logit|",
)
ax.legend(ncol=2, fontsize=8);

- **Heroes move predictions by about 0.6 logit at every stage**, nearly constant. Early on they're the largest single factor (3–8 min: 0.60 vs 0.19 for net worth); from 15 minutes net worth takes over and peaks around 30 minutes (1.1).
- **Damage stats grow steadily** (player damage 0.16 → 0.50); levels matter most mid-game (0.35 at 26–35 min) and fade late.
- **Buildings matter after 25 minutes:** shield generators, barracks and the Titan reach 0.15–0.22.
- `t` has a non-zero share even though it isn't a diff: alone it can't favour either team, but SHAP assigns part of every interaction (e.g. "a lead *at minute 40*") to each feature involved.

## 7. Hero weights (LR + heroes)

The logistic model's hero weights are easier to read. Rung 6 lets each hero's weight change with game time; here at 10 minutes, against the raw win rates:

In [ ]:
lr = saved["baseline_heroes"]
print(f"LR + heroes uses rung {lr['rung']}")
curves = weight_curves(lr["model"], np.array([600, 1200, 2400]))
hero_curves = curves[[c for c in curves.columns if c.startswith("hero_")]].T
hero_curves.index = [int(c.split("_")[1]) for c in hero_curves.index]
hero_curves.columns = ["10 min", "20 min", "40 min"]
compare = hero_curves.join(rates.win_rate.rename("raw win rate"))
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(compare["raw win rate"], compare["10 min"])
for h, row in compare.iterrows():
    ax.annotate(hero_name(h), (row["raw win rate"], row["10 min"]), fontsize=6)
ax.axhline(0, color="grey", lw=0.8)
ax.axvline(0.5, color="grey", lw=0.8)
ax.set(
    title="Hero weight at 10 min (LR + heroes) vs raw win rate (train)",
    xlabel="win rate of the team with the hero",
    ylabel="weight on the logit",
)
print(f"correlation at 10 min: {compare['10 min'].corr(compare['raw win rate']):.3f}")
compare["40 − 10 min"] = compare["40 min"] - compare["10 min"]
compare.rename(index=hero_name).sort_values("40 − 10 min").round(3).iloc[[0, 1, 2, -3, -2, -1]]

The weights follow raw win rates closely (correlation ~0.8), which is reassuring: the model learns from the same signal, after accounting for the game state.

The table shows the heroes whose weight changes most between 10 and 40 minutes: some are **early-game heroes** (strong at 10 minutes, neutral or worse by 40) and some **scale** (weak early, strong late). Since every weight is "given the current stats", a scaling hero is one whose team wins more than its stats suggest late in the game. With only ids, we can't name them here; with names, they're worth checking against what players say.

## 8. Win-probability curves

The same three validation matches as notebook 03, baseline vs final model:

In [ ]:
rng = np.random.default_rng(1)  # the same three matches as notebook 03
picked = rng.choice(val.match_id.unique(), 3, replace=False)
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5), sharey=True)
for ax, mid in zip(axes, picked, strict=True):
    rows = val.match_id.to_numpy() == mid
    g_rows = val[rows]
    for c in dict.fromkeys(["baseline", final["choice"]]):
        ax.plot(g_rows.t / 60, preds[c][rows], marker="o", label=DESCRIBE[c])
    ax.axhline(0.5, color="grey", ls="--")
    winner = "Team0" if g_rows[LABEL].iloc[0] else "Team1"
    ax.set(title=f"match {mid}: {winner} won", xlabel="minute", ylim=(0, 1))
axes[0].set_ylabel("P(Team0 wins)")
axes[0].legend(fontsize=8);

The final model starts further from 50% (it knows the drafts) and then follows the same shape as the baseline: the game state soon dominates.

## 9. Test results

Everything above was decided on validation. `python -m deadlock_predictor.models.refit` refit the baseline and the final model on train + validation, and `python -m deadlock_predictor.models.evaluate_test` scored them **once** on the test split: the newest 15% of matches, which no decision has touched. It refuses to run a second time without `--force`, because looking again after changing something would make test a second validation set.

Each model is scored twice: refit on train + validation (what we'd deploy, the main score), and trained on train only, as validated (to show what refitting gained).

In [ ]:
TEST = MODELS / "test_metrics.json"
if not TEST.exists():
    print("Not evaluated yet: python -m deadlock_predictor.models.evaluate_test")
else:
    test = json.loads(TEST.read_text())
    rows = {}
    for c, r in test["cells"].items():
        for split, label in [
            ("validation", "validation (train-only fit)"),
            ("test_train_only", "test, train-only fit"),
            ("test", "test, refit on train + validation"),
        ]:
            rows[(DESCRIBE[c], label)] = {k: r[split][k] for k in ("log_loss", "brier", "ece")}
    display(pd.DataFrame(rows).T.round(4))
    fig, ax = plt.subplots()
    for c, r in test["cells"].items():
        st = pd.DataFrame(r["test_by_time"]).set_index("t")
        ax.plot(st.index / 60, st.log_loss, marker=".", label=f"{DESCRIBE[c]}, test (refit)")
        val_st = by_time[c]
        ax.plot(val_st.index / 60, val_st.log_loss, ls=":", label=f"{DESCRIBE[c]}, validation")
    ax.axhline(np.log(2), color="grey", ls="--")
    ax.set(title="Log loss by game time: validation vs test", xlabel="minute", ylabel="log loss")
    ax.legend(fontsize=8)

In [ ]:
if TEST.exists():
    # After the fact, changing no decision: matches with a hero no model saw (hero 87).
    refits = {c: joblib.load(MODELS / f"{c}_refit.joblib") for c in test["cells"]}
    test_cols = sorted({col for s in refits.values() for col in snapshot_columns(s["columns"])})
    test_df = load_split("test", test_cols)
    y_test = test_df[LABEL].to_numpy()
    seen = set(refits[final["choice"]]["hero_ids"])
    unseen = sorted(set(hero_table.hero_id[hero_table.match_id.isin(test_df.match_id)]) - seen)
    has_new = test_df.match_id.isin(hero_table.match_id[hero_table.hero_id.isin(unseen)]).to_numpy()
    print(f"test heroes the refit models never saw: {[hero_name(h) for h in unseen]}")
    rows = {}
    for label, sel in [("with an unseen hero", has_new), ("without", ~has_new)]:
        row = {"matches": test_df.match_id[sel].nunique()}
        for c, s in refits.items():
            row[DESCRIBE[c]] = scores(y_test[sel], predict_saved(s, test_df[sel]))["log_loss"]
        rows[label] = row
    split_table = pd.DataFrame(rows).T
    split_table["heroes' gain"] = (
        split_table[DESCRIBE["baseline"]] - split_table[DESCRIBE[final["choice"]]]
    )
    display(split_table.round(4))

**Test log loss** (59,707 matches, Oct 19–24):

| | validation | test, train-only fit | **test, refit** |
|---|---|---|---|
| baseline (rung 3) | 0.5489 | 0.5474 | **0.5472** |
| LightGBM + heroes | 0.5183 | 0.5274 | **0.5232** |

- **The final model beats the baseline on test by 0.024** log loss (Brier 0.1874 → 0.1775, accuracy 70.3% → 72.5%). A real gain, but smaller than on validation (0.031).
- **The baseline held up perfectly:** its test score even edges its validation score. Game-state signals (souls, buildings) are stable within a patch.
- **Why the hero gain shrank: a hero released in the test period.** Hero 87 came out on Oct 20 and is in 54% of test matches; no model ever saw it. In test matches *without* it, heroes still save 0.030, the same as on validation; in matches with it, only 0.019. The model is fine; the roster changed under it.
- **Refitting paid off:** 0.5274 → 0.5232 for the hero model, and its ECE fell from 0.018 to 0.011. The newer matches (and a weight for hero 86) partly corrected the drift. The baseline barely changed, as expected: it doesn't use heroes.
- **Calibration is still the weak spot:** the same overconfidence as on validation (predicted 25% → observed 27%), worse in matches with the unseen hero.

This is an honest estimate because nothing was chosen on test: the model, its settings, the refit and even this split of the comparison were decided before the test was read (the with/without-unseen-hero split is analysis after the fact and changes nothing).

## 10. Summary → Milestone 5

| model | validation | test |
|---|---|---|
| always 50% | 0.6931 | 0.6931 |
| baseline: LR, game state × time | 0.5489 | 0.5472 |
| **final: LightGBM + heroes (refit)** | **0.5183** | **0.5232** |

What we learned:
- **What matters is information, not model complexity.** Heroes gave −0.030 log loss; switching from a linear model to boosted trees gave −0.001.
- **The game changes faster than the data says:** hero strength drifted within one patch, and two heroes were released in 8 days. A deployed model needs **regular retraining** on recent matches and a way to handle new heroes, e.g. a shared "new hero" column, or shrinking unknown heroes toward average.
- **Calibration is the next thing to fix** for the hero model: a recency-weighted fit, or a calibration step fitted on recent matches and checked on a later period.

For Milestone 5 (win-probability curves): use the final model, retrained on all matches up to the present, and keep the baseline as the reference. The test period is now used; any new comparison needs matches newer than Oct 24.